# Experiments on Liverpool Ion Switching Dataset

In [1]:
import numpy as np
import pandas as pd
import lightning as pl
import torch
from torch.utils.data import DataLoader
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint, RichProgressBar
from lightning.pytorch.callbacks.progress.rich_progress import RichProgressBarTheme
from lightning.pytorch.loggers import TensorBoardLogger
# from lightning.pytorch.strategies import DeepSpeedStrategy
import warnings
warnings.filterwarnings('ignore')

pl.seed_everything(42)

Seed set to 42


42

In [2]:
train_df = pd.read_csv('../../data/liverpool-ion-switching/train.csv')
train_df.head()

,time,signal,open_channels
0,0.0001,-2.7600,0
1,0.0002,-2.8557,0
2,0.0003,-2.4074,0
3,0.0004,-3.1404,0
4,0.0005,-3.1525,0


In [3]:
test_df = pd.read_csv('../../data/liverpool-ion-switching/test.csv')
test_df.head()

,time,signal
0,500.0001,-2.6498
1,500.0002,-2.8494
2,500.0003,-2.8600
3,500.0004,-2.4350
4,500.0005,-2.6155


In [4]:
class LiverpoolIonSwitchingDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.Series, sequence_length = 10):
        super().__init__()
        self.df = df
        self.sequence_length = sequence_length

    def __len__(self):
        return self.df.shape[0] - self.sequence_length - 1

    def __getitem__(self, idx):
        signal = self.df.iloc[idx:idx + self.sequence_length, 1].values
        label = self.df.iloc[idx + self.sequence_length - 1, 2]
        return signal, label

In [5]:
class LiverpoolIonSwitchingDataModule(pl.LightningDataModule):
    def __init__(self, train_data, test_data,
                 batch_size: int = 32,
                 train_split: float = 0.8,
                 sequence_length: int = 10,
                 num_workers: int = 4):
        super().__init__()
        self.train_data = train_data
        self.test_data = test_data
        self.batch_size = batch_size
        self.train_split = train_split
        self.sequence_length = sequence_length
        self.num_workers = num_workers
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def setup(self, stage=None):
        train_dataset = LiverpoolIonSwitchingDataset(self.train_data, sequence_length=self.sequence_length)
        test_dataset = LiverpoolIonSwitchingDataset(self.test_data, sequence_length=self.sequence_length)

        train_size = int(self.train_split * len(train_dataset))
        val_size = len(train_dataset) - train_size
        self.train_dataset, self.val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])
        self.test_dataset = test_dataset

    def train_dataloader(self):
        return DataLoader(self.train_dataset,
                          batch_size=self.batch_size,
                          shuffle=True,
                          num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val_dataset,
                          batch_size=self.batch_size,
                          shuffle=False,
                          num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_dataset,
                          batch_size=self.batch_size,
                          shuffle=False,
                          num_workers=self.num_workers)

In [6]:
dm = LiverpoolIonSwitchingDataModule(train_df, test_df,
                                     batch_size=8,
                                     sequence_length=1000,
                                     num_workers=0)

In [7]:
# test datamodule
dm.setup()

train_loader = dm.train_dataloader()

for x, y in train_loader:
    print(x.size(), y.size())
    break

torch.Size([8, 1000]) torch.Size([8])


In [8]:
train_df.shape

(5000000, 3)

In [9]:
from models.wavenet import WaveNet

In [11]:
model = WaveNet(in_channels=1, out_channels=1, kernel_size=3, num_blocks=1, num_layers=5, output_size=train_df['open_channels'].max() + 1)

In [12]:
# set deterministic for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

progress_bar = RichProgressBar(  ## wip
    theme=RichProgressBarTheme(
        description="green_yellow",
        progress_bar="green1",
        progress_bar_finished="green1",
        batch_progress="green_yellow",
        time="grey82",
        processing_speed="grey82",
        metrics="grey82",
    ))

logger = TensorBoardLogger("tensorboard_logs", name=f"wavenet_liverpool")

In [13]:
# init trainer
trainer = pl.Trainer(logger=logger,
                     # strategy='ddp',
                     accelerator='gpu' if str(device) == 'cuda' else 'cpu',
                     devices=1, max_epochs=10,
                     check_val_every_n_epoch=1,
                     callbacks=[ModelCheckpoint(mode='min', monitor='val_loss',
                                                filename='vae-{epoch:02d}-{val_loss:.2f}',
                                                # dirpath=os.path.join(config.CHECKPOINT_DIR, 'vae'),
                                                verbose=False, save_last=True, save_top_k=5,
                                                save_on_train_epoch_end=True),
                                LearningRateMonitor(logging_interval='epoch'),
                                progress_bar])

torch.set_float32_matmul_precision("medium")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [15]:
trainer.fit(model, datamodule=dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type                ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ causal_conv1d  │ CausalDilatedConv1D │      3 │ train │
│ 1 │ residual_stack │ ResidualStack       │     35 │ train │
│ 2 │ dense          │ DenseLayer          │      1 │ train │
└───┴────────────────┴─────────────────────┴────────┴───────┘

Trainable params: 39                                                                                               
Non-trainable params: 0                                                                                            
Total params: 39                                                                                                   
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 43                                                                                          
Modules in eval mode: 0

Output()


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined